Experiment 1: Does our validation strategy give trustworthy estimates?

In [1]:
import pandas as pd

In [2]:
df = pd.read_csv("../data/clean/train.csv")
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 844392 entries, 0 to 844391
Data columns (total 9 columns):
 #   Column         Non-Null Count   Dtype
---  ------         --------------   -----
 0   Store          844392 non-null  int64
 1   DayOfWeek      844392 non-null  int64
 2   Date           844392 non-null  str  
 3   Sales          844392 non-null  int64
 4   Customers      844392 non-null  int64
 5   Open           844392 non-null  int64
 6   Promo          844392 non-null  int64
 7   StateHoliday   844392 non-null  str  
 8   SchoolHoliday  844392 non-null  int64
dtypes: int64(7), str(2)
memory usage: 58.0 MB


In [3]:
df['Date'] = pd.to_datetime(df['Date'])
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 844392 entries, 0 to 844391
Data columns (total 9 columns):
 #   Column         Non-Null Count   Dtype         
---  ------         --------------   -----         
 0   Store          844392 non-null  int64         
 1   DayOfWeek      844392 non-null  int64         
 2   Date           844392 non-null  datetime64[us]
 3   Sales          844392 non-null  int64         
 4   Customers      844392 non-null  int64         
 5   Open           844392 non-null  int64         
 6   Promo          844392 non-null  int64         
 7   StateHoliday   844392 non-null  str           
 8   SchoolHoliday  844392 non-null  int64         
dtypes: datetime64[us](1), int64(7), str(1)
memory usage: 58.0 MB


In [4]:
df.head()

,Store,DayOfWeek,Date,Sales,Customers,Open,Promo,StateHoliday,SchoolHoliday
0,1,5,2015-07-31,5263,555,1,1,0,1
1,2,5,2015-07-31,6064,625,1,1,0,1
2,3,5,2015-07-31,8314,821,1,1,0,1
3,4,5,2015-07-31,13995,1498,1,1,0,1
4,5,5,2015-07-31,4822,559,1,1,0,1


In [5]:
df['Open'].unique()
# drop it
df.drop(columns=["Open"], inplace=True)
df.columns

C:\Users\ecole\AppData\Local\Temp\ipykernel_34248\1618845791.py:3: Pandas4Warning: The inplace keyword in DataFrame.drop is deprecated and will be removed in a future version (PDEP-8).
See https://pandas.pydata.org/docs/dev/whatsnew/v3.1.0.html#deprecation-inplace for more details.
  df.drop(columns=["Open"], inplace=True)


Index(['Store', 'DayOfWeek', 'Date', 'Sales', 'Customers', 'Promo',
       'StateHoliday', 'SchoolHoliday'],
      dtype='str')

## Forecast-origin and target-label construction.


In [6]:
df.head(3)

,Store,DayOfWeek,Date,Sales,Customers,Promo,StateHoliday,SchoolHoliday
0,1,5,2015-07-31,5263,555,1,0,1
1,2,5,2015-07-31,6064,625,1,0,1
2,3,5,2015-07-31,8314,821,1,0,1


In [7]:
df.columns

Index(['Store', 'DayOfWeek', 'Date', 'Sales', 'Customers', 'Promo',
       'StateHoliday', 'SchoolHoliday'],
      dtype='str')

In [8]:

# 1. sort bystore, and order Date chronically
df = df.sort_values(by=["Store", "Date"], ascending=[True, True])
df.head()

,Store,DayOfWeek,Date,Sales,Customers,Promo,StateHoliday,SchoolHoliday
843264,1,3,2013-01-02,5530,668,0,0,1
842155,1,4,2013-01-03,4327,578,0,0,1
841047,1,5,2013-01-04,4486,619,0,0,1
839940,1,6,2013-01-05,4997,635,0,0,1
838815,1,1,2013-01-07,7176,785,1,0,1


In [9]:
df.tail()

,Store,DayOfWeek,Date,Sales,Customers,Promo,StateHoliday,SchoolHoliday
5564,1115,1,2015-07-27,10712,608,1,0,1
4451,1115,2,2015-07-28,8093,500,1,0,1
3338,1115,3,2015-07-29,7661,473,1,0,1
2225,1115,4,2015-07-30,8405,502,1,0,1
1112,1115,5,2015-07-31,8680,538,1,0,1


In [10]:
assert not df.duplicated(['Store', 'Date']).any()

In [11]:
# 2. generate forcast origins
# for each store, i consider each date a potential forcast origin
# only valid if it has

origin_df = df[["Store", "Date"]].copy()
origin_df = origin_df.rename(columns={"Date": "origin_date"})
origin_df.head()

,Store,origin_date
843264,1,2013-01-02
842155,1,2013-01-03
841047,1,2013-01-04
839940,1,2013-01-05
838815,1,2013-01-07


In [12]:
# 3. generate target dates for each horizon

training_rows = []
horizons = [1, 7, 14, 21, 28]

for horizon in horizons:
    temp = origin_df.copy()

    temp['forcast_horizon'] = horizon
    temp["target_date"] = temp["origin_date"] + pd.Timedelta(days=horizon)
    
    training_rows.append(temp)

forcast_df = pd.concat(training_rows, ignore_index=True)
forcast_df.head()

,Store,origin_date,forcast_horizon,target_date
0,1,2013-01-02,1,2013-01-03
1,1,2013-01-03,1,2013-01-04
2,1,2013-01-04,1,2013-01-05
3,1,2013-01-05,1,2013-01-06
4,1,2013-01-07,1,2013-01-08


In [13]:
forcast_df.loc[forcast_df['forcast_horizon'] == 7]

,Store,origin_date,forcast_horizon,target_date
844392,1,2013-01-02,7,2013-01-09
844393,1,2013-01-03,7,2013-01-10
844394,1,2013-01-04,7,2013-01-11
844395,1,2013-01-05,7,2013-01-12
844396,1,2013-01-07,7,2013-01-14
...,...,...,...,...
1688779,1115,2015-07-27,7,2015-08-03
1688780,1115,2015-07-28,7,2015-08-04
1688781,1115,2015-07-29,7,2015-08-05
1688782,1115,2015-07-30,7,2015-08-06


In [14]:
forcast_df.shape

(4221960, 4)

In [15]:
origin_df.shape

(844392, 2)

horizon length=5
origion length = 844,392
training rows should be
844,392 * 5 = 4,221,960

forcast_df is correct in length
for each horizon:
    for each store forcast_horizon = horizon(e.g 1)
    repeat for each date in a store
    like store 1 across all date set forcat_horizon = horizon(e.g 1)
    when all date has finish
    duplicate store 1 across all date again this time set forcat_horizon = the next horizon(e.g 7)
    until all horizon finish


In [16]:
# 4th step to be continued...

In [17]:
forcast_df.head()

,Store,origin_date,forcast_horizon,target_date
0,1,2013-01-02,1,2013-01-03
1,1,2013-01-03,1,2013-01-04
2,1,2013-01-04,1,2013-01-05
3,1,2013-01-05,1,2013-01-06
4,1,2013-01-07,1,2013-01-08


In [18]:
forcast_df['target_date'].duplicated().any()
# True. okay fair, different stores can have same origin date thus targeting same target_date

np.True_

In [19]:
forcast_df.loc[forcast_df['target_date'].duplicated(keep=False)].sort_values('target_date')

,Store,origin_date,forcast_horizon,target_date
515314,682,2013-01-01,1,2013-01-02
400916,530,2013-01-01,1,2013-01-02
387281,512,2013-01-01,1,2013-01-02
194139,259,2013-01-01,1,2013-01-02
425029,562,2013-01-01,1,2013-01-02
...,...,...,...,...
3713731,444,2015-07-31,28,2015-08-28
4185343,1067,2015-07-31,28,2015-08-28
3712954,443,2015-07-31,28,2015-08-28
3719188,451,2015-07-31,28,2015-08-28


In [20]:
forcast_df.duplicated(subset=['Store', 'target_date']).any()

np.True_

In [28]:
forcast_df.loc[
    forcast_df.duplicated(subset=["Store", "target_date"], keep=False)
].sort_values(['Store', 'target_date']).head(15)

,Store,origin_date,forcast_horizon,target_date
5,1,2013-01-08,1,2013-01-09
844392,1,2013-01-02,7,2013-01-09
6,1,2013-01-09,1,2013-01-10
844393,1,2013-01-03,7,2013-01-10
7,1,2013-01-10,1,2013-01-11
844394,1,2013-01-04,7,2013-01-11
8,1,2013-01-11,1,2013-01-12
844395,1,2013-01-05,7,2013-01-12
10,1,2013-01-14,1,2013-01-15
844397,1,2013-01-08,7,2013-01-15


- After checking for duplicate `target_date` values, I checked for duplicates using both `Store` and `target_date`.
- Found duplicate combinations.
- Looking at the rows, the same store can have multiple forecasts targeting the same date, but generated from different origin dates and forecast horizons.
- For example, Store 1 has two forecasts targeting 2013-01-09:
  - Origin date: 2013-01-08, forecast horizon: 1
  - Origin date: 2013-01-02, forecast horizon: 7

This is expected because the same target date can be forecast from different points in time. These are different forecasting origins, not necessarily duplicate records.

A unique forcast obseration is identified by:
`Store + origin_date + forecast_horizon`

In [29]:
forcast_df.shape

(4221960, 4)

In [30]:
# 4. lookup the actual target sales
sales_lookup = df[["Store", "Date", "Sales"]].rename(
    columns={"Date": "target_date", "Sales": "target_sales"}
)

forcast_df = forcast_df.merge(
    sales_lookup,
    on=["Store", "target_date"],
    how="left",
    validate="many_to_one",  # many rows from forcast all map to 1 sales_lookup target_date
)
forcast_df.head()

,Store,origin_date,forcast_horizon,target_date,target_sales
0,1,2013-01-02,1,2013-01-03,4327.0
1,1,2013-01-03,1,2013-01-04,4486.0
2,1,2013-01-04,1,2013-01-05,4997.0
3,1,2013-01-05,1,2013-01-06,NaN
4,1,2013-01-07,1,2013-01-08,5580.0


In [46]:
df.loc[df['Store']==1].head(7)

,Store,DayOfWeek,Date,Sales,Customers,Promo,StateHoliday,SchoolHoliday
843264,1,3,2013-01-02,5530,668,0,0,1
842155,1,4,2013-01-03,4327,578,0,0,1
841047,1,5,2013-01-04,4486,619,0,0,1
839940,1,6,2013-01-05,4997,635,0,0,1
838815,1,1,2013-01-07,7176,785,1,0,1
837710,1,2,2013-01-08,5580,654,1,0,1
836605,1,3,2013-01-09,5471,626,1,0,1


In [31]:
forcast_df.isna().sum()

Store                   0
origin_date             0
forcast_horizon         0
target_date             0
target_sales       350134
dtype: int64

yes, some Store don't have a recorded sale for some target date
what to do?
i don't think i should treat it as missing value and impute with something
becasue truely there is no record for it
solution:
remove invalide forcast origin


In [32]:

forcast_df.loc[forcast_df["origin_date"] == "2013-01-05"].sort_values(
    ["Store", 'forcast_horizon']
).head(15)

,Store,origin_date,forcast_horizon,target_date,target_sales
3,1,2013-01-05,1,2013-01-06,NaN
844395,1,2013-01-05,7,2013-01-12,4952.0
1688787,1,2013-01-05,14,2013-01-19,5182.0
2533179,1,2013-01-05,21,2013-01-26,5598.0
3377571,1,2013-01-05,28,2013-02-02,5970.0
784,2,2013-01-05,1,2013-01-06,NaN
845176,2,2013-01-05,7,2013-01-12,2630.0
1689568,2,2013-01-05,14,2013-01-19,2399.0
2533960,2,2013-01-05,21,2013-01-26,2205.0
3378352,2,2013-01-05,28,2013-02-02,2956.0


- group rows with same `Store` and `origin_date`
example:
```
Store 1, 2013-01-02
Store 1, 2013-01-02	
Store 1, 2013-01-05
```

- select `target_sales` and count
- For each group, count how many non-null `target_sales` values it contains.
- `transform()` returns a result(the count of `target_sales`) for every row
- sticking to the above example:
```
Store 1, 2013-01-02, `target_sales` count = 5
Store 1, 2013-01-02, `target_sales` count = 5
Store 1, 2013-01-05, `target_sales` count = 4
```

In [33]:
# keep only forcast origins with all 5 targets observations
# becasue we want to evaluate forecasts at horizons 1, 7, 14, 21 and 28, we need complete observations for all five horizons.

valid_origins = (
    forcast_df.groupby(['Store', 'origin_date'])['target_sales']
    .transform('count') == len(horizons)
)
valid_origins   

0           True
1           True
2           True
3          False
4           True
           ...  
4221955    False
4221956    False
4221957    False
4221958    False
4221959    False
Name: target_sales, Length: 4221960, dtype: bool

In [34]:
forcast_df = forcast_df[valid_origins].copy()

forcast_df = forcast_df.sort_values(
    ['Store', 'origin_date', 'forcast_horizon']
)
forcast_df.shape

(2808265, 5)

In [35]:
forcast_df.head(25)

,Store,origin_date,forcast_horizon,target_date,target_sales
0,1,2013-01-02,1,2013-01-03,4327.0
844392,1,2013-01-02,7,2013-01-09,5471.0
1688784,1,2013-01-02,14,2013-01-16,4008.0
2533176,1,2013-01-02,21,2013-01-23,5578.0
3377568,1,2013-01-02,28,2013-01-30,4601.0
1,1,2013-01-03,1,2013-01-04,4486.0
844393,1,2013-01-03,7,2013-01-10,4892.0
1688785,1,2013-01-03,14,2013-01-17,4044.0
2533177,1,2013-01-03,21,2013-01-24,5195.0
3377569,1,2013-01-03,28,2013-01-31,4709.0


In [ ]:
# df.loc[df["Date"] == "2013-01-25" and df['Store'] == 1]
df.loc[df["Date"] == "2013-01-25"]
# store 1 sals on 2013-01-04 = 5586
# correct with forcast_df row index 2533178

,Store,DayOfWeek,Date,Sales,Customers,Promo,StateHoliday,SchoolHoliday
821081,1,5,2013-01-25,5586,676,1,0,0
821082,2,5,2013-01-25,4589,522,1,0,0
821083,3,5,2013-01-25,7565,773,1,0,0
821084,4,5,2013-01-25,9730,1409,1,0,0
821085,5,5,2013-01-25,4960,564,1,0,0
...,...,...,...,...,...,...,...,...
822184,1111,5,2013-01-25,5114,485,1,0,0
822185,1112,5,2013-01-25,13661,1064,1,0,0
822186,1113,5,2013-01-25,6714,778,1,0,0
822187,1114,5,2013-01-25,19446,2838,1,0,0


In [36]:
forcast_df.isna().sum()

Store              0
origin_date        0
forcast_horizon    0
target_date        0
target_sales       0
dtype: int64

validation question is:

> How should I evaluate a four-week forecasting model to establish whether its predictions will remain reliable when predicting future harvests?

I decided to evaluate at 1, 7, 14, 21 and 28 days ahead.

To investigate how forecast accuracy changes with prediction horizon.

| Horizon | What you're evaluating         |
| ------- | ------------------------------ |
| 1 day   | Predicting tomorrow's sales    |
| 7 days  | Predicting sales 7 days ahead  |
| 14 days | Predicting sales 14 days ahead |
| 21 days | Predicting sales 21 days ahead |
| 28 days | Predicting sales 28 days ahead |

For each horizon, compare the error metric


| Horizon | MAE | RMSE | WAPE |
| ------- | --- | ---- | ---- |
| 1 day   | ... | ...  | ...  |
| 7 days  | ... | ...  | ...  |
| 14 days | ... | ...  | ...  |
| 21 days | ... | ...  | ...  |
| 28 days | ... | ...  | ...  |

Then plot horizon against forecast error.


> How should I evaluate a four-week forecasting model to establish whether its predictions will remain reliable when predicting future harvests?

1st think is to sort by store and date.
such that sales is in chronical order for each store.

A forecast origin is the date at which you stand and ask:
"What will sales be in the future?"

ex: constructing training dataset
i stand at jan 1
jan 1 is the forcast origin

i want 5 predictions: 1 day ahead, 7 days ahead, 14, 21, and28 days ahead predictions
theses are my forcast horizon for Jan 1 forcast origin

We repeat this process at different forecast origins.

If you have 1,000 valid forecast origins and five horizons, you could have approximately 5,000 training examples per store.

I use dates as forcast origin provded there is enough future data for the 5 horizon.


to avoid future leakage
use only information available at forcast origin.

Important: Historical sales features such as lags must only use sales available at the forecast origin. Target-date calendar and promotion features are allowed only when those values are genuinely known in advance.

I am imagining this business makes a new forcast everyday.

Within each store valid forcast origin = t if t+28 <= available date

Validation STrategies to test:
1. Hold-Out train-test split
2. K-Fold cross-validation
    - Group K-Fold group=Store
    - Expanding window(Walk-Forward)
    - Rolling window
3. Out-of-Time(OOT) validation
    - static holdout

- i think 3 is the time-awareness of 1
so i would remove 1 and leave 3
- i think OOT validation is the umberlla name for (expanding window, rolling window, andstatic holdout)
statc holdout train on a fixed historic window and validate on the next window. once nothing is moving. almost like hold-out train-tes split just not random spliting.
- group k-fold becos store is an entity
then also needs time aware training n validating.
or is this the method for experiment 3 Model architecture strategy | Should we train one global model or individual models per store?




model = xgboost
target = Sales
repression problem

after splitting

making the data model ready
1. Date column
    - extract: year, month
    - drop Date
2. StateHoliday column
    - one hot encode since only 4 unique values['0', 'a', 'b', 'c'] so 3 new columns, dropping 1('0') if it's not a or b or c it's 0
3. Customers colums
    - it represents the number of customers on a given day
    - prediction is produced before the day happens
    - drop becos wouldn't be available

feature engineering:
- what to create?